# Medical Agent — Colab runner
This notebook runs the Hugging Face intake conversation, separate patient history, the CPU Delphi demonstration model, four RAG stores and the final verdict model. It is a research prototype, not a diagnostic or treatment tool. Do not enter identifying information.

In [ ]:
%pip install -q "transformers>=4.45,<5" "accelerate>=1,<2" "safetensors>=0.4" "pydantic>=2.8,<3" "scikit-learn>=1.4,<2"

In [ ]:
# Upload medical_agent_colab.zip when the file chooser appears.
from google.colab import files
from pathlib import Path
import os, shutil, zipfile

upload = files.upload()
archive_name = next((name for name in upload if name.lower().endswith('.zip')), None)
if archive_name is None:
    raise RuntimeError('Upload the project ZIP file.')
project_root = Path('/content/medical_agent')
if project_root.exists():
    shutil.rmtree(project_root)
project_root.mkdir(parents=True)
with zipfile.ZipFile(archive_name) as archive:
    archive.extractall(project_root)
if not (project_root / 'src').exists():
    candidates = [path.parent for path in project_root.rglob('requirements.txt') if (path.parent / 'src').exists()]
    if not candidates:
        raise RuntimeError('The ZIP does not contain the medical-agent source tree.')
    project_root = candidates[0]
os.chdir(project_root)
print('Project ready at', project_root)

In [ ]:
import torch
from src.interfaces.colab_chat import ColabMedicalSession

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Qwen device:', device)
print('Delphi training remains CPU-compatible; the model has approximately 2M parameters.')
session = ColabMedicalSession(device=device, train_demo_delphi=True, delphi_epochs=3, debug=True)
print('Session initialized. Qwen downloads lazily on the first answer.')

In [ ]:
from src.interfaces.colab_chat import run_interactive_session
run_interactive_session(session)

Run the previous cell again after creating a new `ColabMedicalSession` if you want a clean conversation. The bundled Delphi trajectories are source-derived software fixtures with artificial ages; their scores have no clinical predictive meaning.